### Historical Weather Data for New Mexico Coal Generating Stations

For this project, I'm using the Open-Meteo Python API to pull historical weather data for three coal-fired power generating stations in New Mexico. The API provides current and historical weather data for locations using latitude and longitude.

To run my notebook, I first need to install the Open-Meteo Python client and NumPy. The client uses NumPy arrays to return the daily weather measurements, so I need both packages to work with the data.

In [ ]:
# Install the Open-Meteo Python API and NumPy 
!pip install openmeteo-requests
!pip install numpy

Next, I import `openmeteo_requests` to request weather data, `csv` to create my CSV files, and `datetime` to work with dates. I'll use these tools to collect daily records and calculate annual totals and averages for each location.

In [ ]:
# Import requests for historical weather data
import openmeteo_requests
# Import tools for working with dates and writing CSV files
import datetime
import csv

### Selecting locations and defining the date range

For this exercise, I chose three coal-fired power generating stations in New Mexico that are either still operating or have been retired: San Juan Generating Station, Four Corners Generating Station, and Escalante Generating Station.

I chose these because I work with environmental data in my day job, particularly location-based data in the energy sector, and I thought it would be interesting to look at the historical weather conditions around these plants.

I put the locations into a list called `sites`, with a dictionary for each location containing its name, latitude, and longitude. That lets me loop through the sites later instead of writing a separate request for each one.

I also define `start_date` and `end_date` as variables. I'm using a 20-year period because that is a common lookback period in some of the environmental regulatory work I'm familiar with. My records run from January 1, 2006, through December 31, 2025. I ended with 2025 because it is the most recent complete calendar year, rather than using a partial year.

In [ ]:
# Add selected locations/sites to a list of dictionaries that we can loop over
sites = [
    {'name': 'San Juan Generating Station', 'latitude': 36.80431827352674, 'longitude': -108.44075687014612},
    {'name': 'Four Corners Generating Station', 'latitude': 36.686098340709236, 'longitude': -108.48060240421098},
    {'name': 'Escalante Generating Station', 'latitude': 35.41561122123407, 'longitude': -108.08289423182069},
]

# Define range of years by start/end date to get a 20 year record 
start_date = datetime.date(2006, 1, 1)
end_date = datetime.date(2025, 12, 31)

### Defining the weather variables

Open-Meteo has a wide range of weather variables available. I chose a smaller set focused on precipitation, evapotranspiration, soil moisture, relative humidity, and maximum temperature. I wanted to get a sense of how dry these areas are and how much precipitation they receive.

I put the measurements I want into a list called `variables`. Next, I create the Open-Meteo client and define the URL for the Historical Weather API using the Open-Meteo documentation.

As I did in my Basics exercise, I am creating an empty list called `records`. I'll append a dictionary to this list for every site and day returned by the API.

In [ ]:
# Define the set of daily weather variables to use.
variables = [
    'precipitation_sum',
    'et0_fao_evapotranspiration',
    'soil_moisture_0_to_7cm_mean',
    'relative_humidity_2m_mean',
    'temperature_2m_max',
]

# Create the client request and define the url used
openmeteo = openmeteo_requests.Client()
url = 'https://archive-api.open-meteo.com/v1/archive'

# Make an empty list to append each site and day to
records = []

### Requesting and collecting daily weather data

Next, I loop through the locations in my `sites` list and request the weather variables for each one. For each site, I create a `params` dictionary with its latitude and longitude, my start and end dates, and the daily variables I selected.

The Historical Weather API requires dates in `YYYY-MM-DD` format, so I use `.isoformat()` on my date variables. I also select the ERA5 historical weather model because it includes the variables I need and set the time zone to GMT to keep the dates consistent.

I use the Open-Meteo client to make the API request. The request returns a list of responses, and since I'm requesting one site at a time, I use `responses[0]` to get that site's response. Then I use `response.Daily()` and `.ValuesAsNumpy()` to get the daily values for each variable. The variable numbers correspond to the order I listed them in `variables`.

The API also returns a timestamp for the start of the daily data. I use `datetime` to convert that timestamp into a date and store it as `starting_date`.

Then I loop through the daily measurements using `for i in range(len(precipitation))`. The length tells me how many daily records were returned, and `i` starts at zero and increases by one for each record. I add that number of days to `starting_date` using `datetime.timedelta(days=i)`, so each record gets the correct date.

For each day, I create a dictionary containing the location, date, and all five weather measurements. I append that dictionary to `records`. By the end of the loops, I have daily weather records for all three locations across the 20-year period.

In [ ]:
# Request the variables for the given date ranges for each site
for site in sites:
    params = {
        'latitude': site['latitude'],
        'longitude': site['longitude'],
        'start_date': start_date.isoformat(),  # YYYY-MM-DD required by the historical API
        'end_date': end_date.isoformat(),      
        'daily': variables,
        'models': 'era5',
        'timezone': 'GMT',
    }

    # Get the weather data for this location from the API response list
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    daily = response.Daily()

    # Get the daily measurements for each variable in the order requested from the API
    precipitation = daily.Variables(0).ValuesAsNumpy()
    et0 = daily.Variables(1).ValuesAsNumpy()
    soil_moisture = daily.Variables(2).ValuesAsNumpy()
    humidity = daily.Variables(3).ValuesAsNumpy()
    temperature = daily.Variables(4).ValuesAsNumpy()

    # Convert the API's timestamp into a date for the CSV
    starting_date = datetime.datetime.fromtimestamp(
        daily.Time(), datetime.timezone.utc
    ).date()

    # Add a dictionary for each day
    for i in range(len(precipitation)):
        day = starting_date + datetime.timedelta(days=i)
        record = {
            'location': site['name'],
            'date': day.isoformat(),
            'precipitation_mm': float(precipitation[i]),
            'et0_mm': float(et0[i]),
            'soil_moisture_m3_m3': float(soil_moisture[i]),
            'relative_humidity_pct': float(humidity[i]),
            'max_temperature_c': float(temperature[i]),
        }
        records.append(record)


### Saving the daily records

Once I have the daily data, I use `open()` and `csv.DictWriter()` to save it as `nm_coal_weather_daily_2006_2025.csv`. I use `newline=''` to avoid extra blank lines between rows in the CSV.

I define the columns for the location, date, and five measurements. Then I use `writeheader()` to add the column names and `writerows(records)` to write the records I collected. This gives me one row per site per day, with the daily measurements kept at their original precision.

In [ ]:
# Save one row per location and date 
with open('nm_coal_weather_daily_2006_2025.csv', 'w', newline='') as f:
    columns = [
        'location', 'date', 'precipitation_mm', 'et0_mm',
        'soil_moisture_m3_m3', 'relative_humidity_pct', 'max_temperature_c'
    ]
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(records)

### Calculating annual totals and averages

Next, I want to summarize the daily records by year so the results are easier to compare across the three locations. I could use this data to look at year-over-year changes or compare different 10-year periods, but for this exercise I'm stopping at creating annual totals and averages.

I create an empty list called `annual_records` and loop through each site and each year from 2006 through 2025. For each site and year, I make empty lists for the five measurements. Then I loop through the daily `records` and collect the values that match that location and year.

I use `sum()` to calculate the annual precipitation and evapotranspiration totals. For soil moisture, relative humidity, and maximum temperature, I calculate annual averages by dividing the sum of the daily values by the number of values in each list using `len()`. A complete year generally has 365 days, or 366 in a leap year, so using `len()` lets me use the actual number of daily records.

I also subtract annual evapotranspiration from annual precipitation. This gives us a simple way to compare how much precipitation an area receives against how much water the atmosphere could potentially pull from the ground & plantlife. In our case, it helps show how dry these areas can be.

I add each year's results as a dictionary in `annual_records`. Finally, I write those results to `nm_coal_weather_annual_2006_2025.csv` using `csv.DictWriter()`, with one row for each site and year. The data can be used for further analysis and given the precise nature of some of these measurements I chose not to round the values.


In [ ]:
# Keep the annual results in a new list.
annual_records = []

# loop through each location and each year 
for site in sites:
    for year in range(2006, 2026):
        # Collect this site's daily measurements for this year.
        precipitation_values = []
        et0_values = []
        soil_values = []
        humidity_values = []
        temperature_values = []

        for record in records:
            if record['location'] == site['name'] and record['date'][:4] == str(year):
                precipitation_values.append(record['precipitation_mm'])
                et0_values.append(record['et0_mm'])
                soil_values.append(record['soil_moisture_m3_m3'])
                humidity_values.append(record['relative_humidity_pct'])
                temperature_values.append(record['max_temperature_c'])

        # Calculate results only for years with daily records.
        if precipitation_values:
            annual_record = {
                'location': site['name'],
                'year': year,
                'annual_precipitation_mm': sum(precipitation_values),
                'annual_et0_mm': sum(et0_values),
                'mean_soil_moisture_m3_m3': sum(soil_values) / len(soil_values),
                'mean_relative_humidity_pct': sum(humidity_values) / len(humidity_values),
                'mean_max_temperature_c': sum(temperature_values) / len(temperature_values),
            }
            annual_record['precipitation_minus_et0_mm'] = (
                annual_record['annual_precipitation_mm'] - annual_record['annual_et0_mm']
            )
            annual_records.append(annual_record)

# Write one CSV row for each site and year.
with open('nm_coal_weather_annual_2006_2025.csv', 'w', newline='') as f:
    columns = [
        'location', 'year', 'annual_precipitation_mm', 'annual_et0_mm',
        'mean_soil_moisture_m3_m3', 'mean_relative_humidity_pct',
        'mean_max_temperature_c', 'precipitation_minus_et0_mm'
    ]
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(annual_records)
